# Chapter 5: Data Class Builders
## Fluent Python, 2nd Edition -- Distilled Interactive Notebook

> "Data classes are like children. They are okay as a starting point, but to participate as a grownup object, they need to take some responsibility."
> -- *Martin Fowler and Kent Beck*

Related wiki articles: [[data-class-builders-overview]], [[classic-named-tuples]], [[typed-named-tuples]], [[type-hints-101]], [[dataclass-decorator-and-fields]], [[post-init-and-advanced-features]], [[data-class-as-code-smell]], [[pattern-matching-class-instances]]

## TL;DR

- Python provides **three data class builders**: `collections.namedtuple`, `typing.NamedTuple`, and `@dataclasses.dataclass` -- each auto-generates `__init__`, `__repr__`, `__eq__`.
- **namedtuple** and **NamedTuple** produce immutable tuple subclasses; `@dataclass` produces mutable classes by default (`frozen=True` for immutability).
- **Type hints** on data class fields have **zero runtime effect** -- they only populate `__annotations__` for static checkers.
- Use `field(default_factory=list)` for mutable defaults in `@dataclass` -- bare `list` defaults are rejected.
- `__post_init__` runs after the generated `__init__` for validation or computed fields. `ClassVar` and `InitVar` control what becomes a field.

---
## 1. Overview of Data Class Builders

All three builders solve the same problem: generating boilerplate (`__init__`, `__repr__`, `__eq__`) from field declarations. They differ in mutability, syntax, and metaprogramming technique.

| Feature | namedtuple | NamedTuple | @dataclass |
|---|---|---|---|
| Mutable | No | No | Yes (default) |
| Class syntax | No | Yes | Yes |
| Type hints | No | Yes | Yes |
| To dict | `._asdict()` | `._asdict()` | `dataclasses.asdict()` |

See also: [[data-class-builders-overview]]

In [ ]:
# The problem: plain classes require tedious boilerplate
class Coordinate:
    def __init__(self, lat, lon):
        self.lat = lat
        self.lon = lon

moscow = Coordinate(55.76, 37.62)
location = Coordinate(55.76, 37.62)

print("repr:", moscow)             # unhelpful <...object at 0x...>
print("equal?", moscow == location)  # False! compares object IDs
print("values match?", (moscow.lat, moscow.lon) == (location.lat, location.lon))

In [ ]:
# All three builders solve this -- quick comparison
from collections import namedtuple
from typing import NamedTuple
from dataclasses import dataclass

# 1. collections.namedtuple
CoordNT = namedtuple('CoordNT', 'lat lon')

# 2. typing.NamedTuple
class CoordTNT(NamedTuple):
    lat: float
    lon: float

# 3. @dataclass
@dataclass
class CoordDC:
    lat: float
    lon: float

# All three now have proper __repr__ and __eq__
for cls in [CoordNT, CoordTNT, CoordDC]:
    a = cls(55.76, 37.62)
    b = cls(55.76, 37.62)
    print(f"{cls.__name__:12}  repr={a!r:45}  eq={a == b}")

---
## 2. Classic Named Tuples (collections.namedtuple)

A factory function that builds **immutable tuple subclasses** with named fields. Provides `_fields`, `_asdict()`, `_make()`, `_replace()`, and `_field_defaults`.

See also: [[classic-named-tuples]]

In [ ]:
from collections import namedtuple

City = namedtuple('City', 'name country population coordinates')
tokyo = City('Tokyo', 'JP', 36.933, (35.689722, 139.691667))

print("repr:", tokyo)
print("name:", tokyo.name)       # access by name
print("country:", tokyo[1])      # access by index (it's a tuple!)
print("Is tuple?", isinstance(tokyo, tuple))

In [ ]:
# Named tuple special attributes and methods
from collections import namedtuple

City = namedtuple('City', 'name country population coordinates')
Coordinate = namedtuple('Coordinate', 'lat lon')

print("_fields:", City._fields)

# _make: build from iterable (like City(*iterable))
delhi_data = ('Delhi NCR', 'IN', 21.935, Coordinate(28.613889, 77.208889))
delhi = City._make(delhi_data)
print("_make:", delhi)

# _asdict: convert to dict
print("_asdict:", delhi._asdict())

# _replace: create new instance with some fields changed
print("_replace:", delhi._replace(population=28.5))

# defaults (Python 3.7+)
Place = namedtuple('Place', 'name lat lon reference', defaults=['WGS84'])
print("defaults:", Place('Here', 0, 0))
print("_field_defaults:", Place._field_defaults)

---
## 3. Typed Named Tuples (typing.NamedTuple)

Extends classic namedtuples with **class syntax** and **type annotations**. Runtime behavior is identical to `collections.namedtuple`, but carries `__annotations__`.

See also: [[typed-named-tuples]]

In [ ]:
from typing import NamedTuple

class Coordinate(NamedTuple):
    lat: float
    lon: float
    reference: str = 'WGS84'

    def __str__(self):
        ns = 'N' if self.lat >= 0 else 'S'
        we = 'E' if self.lon >= 0 else 'W'
        return f'{abs(self.lat):.1f}\u00b0{ns}, {abs(self.lon):.1f}\u00b0{we}'

moscow = Coordinate(55.756, 37.617)
print("repr:", repr(moscow))
print("str:", moscow)
print("Is tuple?", isinstance(moscow, tuple))
# Note: NamedTuple is NOT a regular class -- issubclass() would raise TypeError
print("Is tuple subclass:", issubclass(Coordinate, tuple))

import typing
print("annotations:", typing.get_type_hints(Coordinate))

---
## 4. Type Hints 101 for Data Classes

Type hints have **zero runtime effect** -- Python reads them at import time to build `__annotations__` but never enforces them. Only external tools like **mypy** check types. The `var: type = value` syntax is how NamedTuple and @dataclass declare fields.

See also: [[type-hints-101]]

In [ ]:
# Type hints have NO runtime effect
from typing import NamedTuple

class Coordinate(NamedTuple):
    lat: float
    lon: float

# Python happily accepts wrong types at runtime!
trash = Coordinate('Ni!', None)
print("No error:", trash)

# Annotations are stored but not enforced
print("__annotations__:", Coordinate.__annotations__)

In [ ]:
# Plain class annotations: only annotated+assigned creates a class attribute
class DemoPlain:
    a: int             # annotation only -> NO class attribute
    b: float = 1.1     # annotation + value -> class attribute
    c = 'spam'         # no annotation -> plain class attribute

print("__annotations__:", DemoPlain.__annotations__)  # only a, b
print("b:", DemoPlain.b)   # 1.1
print("c:", DemoPlain.c)   # 'spam'

try:
    print(DemoPlain.a)
except AttributeError as e:
    print(f"a not found: {e}")  # 'a' is just an annotation, not an attribute

---
## 5. @dataclass Decorator and Field Options

The `@dataclass` decorator accepts options: `init`, `repr`, `eq`, `order`, `frozen`, `unsafe_hash`. The `field()` function provides per-field control: `default_factory`, `repr`, `compare`, `hash`, `metadata`.

See also: [[dataclass-decorator-and-fields]]

In [ ]:
from dataclasses import dataclass

# frozen=True emulates immutability
@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

    def __str__(self):
        ns = 'N' if self.lat >= 0 else 'S'
        we = 'E' if self.lon >= 0 else 'W'
        return f'{abs(self.lat):.1f}°{ns}, {abs(self.lon):.1f}°{we}'

moscow = Coordinate(55.756, 37.617)
print("repr:", repr(moscow))
print("str:", str(moscow))
print("equal?", moscow == Coordinate(55.756, 37.617))

# frozen means we can hash it (useful as dict key or set element)
print("hashable:", hash(moscow))

# But we can't modify it
try:
    moscow.lat = 0
except AttributeError as e:
    print(f"Frozen: {e}")

In [ ]:
from dataclasses import dataclass, field, fields

# field() for mutable defaults and per-field options
@dataclass
class ClubMember:
    name: str
    guests: list[str] = field(default_factory=list)  # NOT [] !
    athlete: bool = field(default=False, repr=False)  # hidden from repr

alice = ClubMember('Alice', ['Bob'])
bob = ClubMember('Bob')

print("alice:", alice)  # athlete not shown in repr
print("bob:", bob)      # guests is a new empty list, not shared

# Inspect fields
for f in fields(ClubMember):
    print(f"  {f.name}: default={f.default!r}, default_factory={f.default_factory!r}")

In [ ]:
# Why bare mutable defaults are rejected
from dataclasses import dataclass

try:
    @dataclass
    class Bad:
        items: list = []  # REJECTED!
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# @dataclass with order=True enables sorting
from dataclasses import dataclass

@dataclass(order=True)
class Student:
    name: str
    grade: float

students = [Student('Charlie', 3.2), Student('Alice', 3.8), Student('Bob', 3.5)]
print("sorted:", sorted(students))

# Comparison is tuple-like: (name, grade)
print("Alice < Bob?", Student('Alice', 3.8) < Student('Bob', 3.5))

---
## 6. Post-init Processing and Advanced Features

`__post_init__` runs after the generated `__init__` for validation or computed fields. `ClassVar` excludes class-level attributes from fields. `InitVar` declares init-only parameters.

See also: [[post-init-and-advanced-features]]

In [ ]:
from dataclasses import dataclass, field

@dataclass
class HackerClubMember:
    name: str
    guests: list[str] = field(default_factory=list)
    handle: str = ''

    def __post_init__(self):
        # Compute default handle from name
        if not self.handle:
            self.handle = self.name.split()[0]
        # Validate uniqueness using class-level set
        if self.handle in HackerClubMember._all_handles:
            raise ValueError(f"handle {self.handle!r} already exists.")
        HackerClubMember._all_handles.add(self.handle)

# Class-level set to track all handles (NOT a dataclass field)
HackerClubMember._all_handles = set()

anna = HackerClubMember('Anna Ravenscroft', handle='AnnaRaven')
print(anna)

leo = HackerClubMember('Leo Rochael')
print(leo)  # handle auto-derived from first name

try:
    leo2 = HackerClubMember('Leo DaVinci')  # handle 'Leo' already taken
except ValueError as e:
    print(f"Duplicate: {e}")

In [ ]:
# InitVar: passed to __post_init__ but NOT stored as instance attribute
from dataclasses import dataclass, field, InitVar

@dataclass
class DatabaseRecord:
    description: str
    db_handle: InitVar[str] = ''  # init-only, not stored
    _resource_id: int = field(init=False, default=0)

    def __post_init__(self, db_handle):
        # Use db_handle for initialization logic
        if db_handle:
            self._resource_id = hash(db_handle) % 10000
        print(f"  (connected to {db_handle!r}, id={self._resource_id})")

rec = DatabaseRecord('Test record', db_handle='prod-db-01')
print("Record:", rec)
print("Has db_handle attr?", hasattr(rec, 'db_handle'))  # False!

---
## 7. Data Class as Code Smell vs Scaffolding

A class with **only data and no behavior** is a code smell (Fowler/Beck) -- it suggests logic that belongs in the class lives elsewhere. But data classes are **legitimate** as:
- **Scaffolding** during early development
- **Intermediate representations** at serialization boundaries (JSON, DB records)

See also: [[data-class-as-code-smell]]

In [ ]:
# From code smell to proper class: add behavior to your data classes
from dataclasses import dataclass

@dataclass
class Vector:
    x: float
    y: float

    # Add behavior -- now it's more than just data!
    @property
    def magnitude(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def normalized(self):
        m = self.magnitude
        return Vector(self.x / m, self.y / m) if m else Vector(0, 0)

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

v = Vector(3, 4)
print(f"{v} magnitude={v.magnitude}")
print(f"normalized: {v.normalized()}")
print(f"{v} + {Vector(1, 1)} = {v + Vector(1, 1)}")

---
## 8. Pattern Matching Class Instances (Python 3.10+)

`match/case` supports **class patterns**: simple (`float(x)`), keyword (`City(continent='Asia')`), and positional (`City('Asia', _, country)`). Data class builders auto-generate `__match_args__`.

See also: [[pattern-matching-class-instances]]

In [ ]:
# Pattern matching with data class instances
from typing import NamedTuple

class City(NamedTuple):
    continent: str
    name: str
    country: str

cities = [
    City('Asia', 'Tokyo', 'JP'),
    City('Asia', 'Delhi', 'IN'),
    City('North America', 'Mexico City', 'MX'),
    City('South America', 'Sao Paulo', 'BR'),
]

# Keyword class pattern
for city in cities:
    match city:
        case City(continent='Asia', name=name, country=country):
            print(f"Asian city: {name}, {country}")
        case City(continent=continent, name=name):
            print(f"Other: {name} in {continent}")

In [ ]:
# Positional matching uses __match_args__ (auto-generated by data class builders)
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

print("__match_args__:", Point.__match_args__)

# Simple class patterns
def classify_point(point):
    match point:
        case Point(0, 0):
            return "origin"
        case Point(0, y):
            return f"on Y axis at y={y}"
        case Point(x, 0):
            return f"on X axis at x={x}"
        case Point(x, y) if x == y:
            return f"on diagonal at ({x}, {y})"
        case Point(x, y):
            return f"at ({x}, {y})"

points = [Point(0, 0), Point(0, 5), Point(3, 0), Point(4, 4), Point(1, 2)]
for p in points:
    print(f"  {p!r:20} -> {classify_point(p)}")

---
## Exercises

Test your understanding of Chapter 5 concepts.

### Exercise 1: namedtuple vs @dataclass
Create a `Book` using both `namedtuple` and `@dataclass` with fields: title (str), author (str), year (int). Show that namedtuple instances are immutable while dataclass instances are mutable.

In [ ]:
# Exercise 1
from collections import namedtuple
from dataclasses import dataclass

# Solution:
BookNT = namedtuple('BookNT', 'title author year')
b1 = BookNT('Fluent Python', 'Ramalho', 2022)
print("namedtuple:", b1)
try:
    b1.year = 2023
except AttributeError as e:
    print(f"Immutable: {e}")

@dataclass
class BookDC:
    title: str
    author: str
    year: int

b2 = BookDC('Fluent Python', 'Ramalho', 2022)
b2.year = 2023  # works!
print(f"dataclass after mutation: {b2}")

### Exercise 2: default_factory
Create a `Team` dataclass with fields `name: str` and `members: list[str]`. Demonstrate that two separate instances get independent member lists.

In [ ]:
# Exercise 2
from dataclasses import dataclass, field

# Solution:
@dataclass
class Team:
    name: str
    members: list[str] = field(default_factory=list)

alpha = Team('Alpha')
bravo = Team('Bravo')
alpha.members.append('Alice')
alpha.members.append('Bob')

print(f"{alpha.name}: {alpha.members}")
print(f"{bravo.name}: {bravo.members}")  # Should be empty!
print(f"Independent lists? {alpha.members is not bravo.members}")

### Exercise 3: __post_init__ Validation
Create a `Temperature` dataclass with a `celsius` field and a `__post_init__` that raises ValueError if celsius < -273.15 (absolute zero).

In [ ]:
# Exercise 3
from dataclasses import dataclass

# Solution:
@dataclass
class Temperature:
    celsius: float

    def __post_init__(self):
        if self.celsius < -273.15:
            raise ValueError(
                f"Temperature {self.celsius}°C is below absolute zero!")

    @property
    def fahrenheit(self):
        return self.celsius * 9/5 + 32

t = Temperature(100)
print(f"{t} -> {t.fahrenheit}°F")

t2 = Temperature(-40)
print(f"{t2} -> {t2.fahrenheit}°F")

try:
    t3 = Temperature(-300)
except ValueError as e:
    print(f"Validation: {e}")

---
## Key Takeaways

1. **Three builders, one goal**: `namedtuple`, `NamedTuple`, and `@dataclass` all auto-generate `__init__`, `__repr__`, `__eq__` from field declarations.
2. **Immutable vs mutable**: namedtuple/NamedTuple produce tuple subclasses (immutable). `@dataclass` is mutable by default; use `frozen=True` for immutability.
3. **Type hints are not enforced** at runtime. They build `__annotations__` for static checkers (mypy, IDE).
4. **Mutable defaults are rejected** by `@dataclass`. Use `field(default_factory=list)` instead of `[]`.
5. **`__post_init__`** is your hook for validation and computed fields after `__init__`.
6. **`ClassVar`** excludes class attributes from dataclass fields; **`InitVar`** creates init-only params passed to `__post_init__`.
7. **Data classes as code smell**: a class with data but no behavior suggests misplaced logic. Add methods to graduate from "data holder" to proper object.
8. **Pattern matching** with class instances uses `__match_args__` (auto-generated) for positional destructuring.

See also: [[data-class-builders-overview]], [[classic-named-tuples]], [[typed-named-tuples]], [[type-hints-101]], [[dataclass-decorator-and-fields]], [[post-init-and-advanced-features]], [[data-class-as-code-smell]], [[pattern-matching-class-instances]]